# DX 704 Week 4 Project - Linear Contextual Bandits

This week's project will <font color = 'salmon'> test the learning speed of *linear contextual bandits* compared to unoptimized approaches.</font>
You will start with building a preference data set for evaluation, and then implement different variations of *LinUCB* and visualize how fast they learn the preferences.


The full project description, a template notebook and supporting code are available on GitHub: [Project 4 Materials](https://github.com/bu-cds-dx704/dx704-project-04).

<font color = 'red'> FINAL SCORE: 80/80 </font>


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Imports

In [380]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split


sns.set_theme(font_scale=0.8) 
# plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.titlesize']  = 10
plt.rcParams['axes.labelsize']  = 8
plt.rcParams['lines.linewidth'] = 0.5
# plt.rcParams['lines.markersize'] = 3
plt.rcParams['axes.edgecolor']  = 'gray'
plt.rcParams['xtick.color']     = 'gray'
plt.rcParams['ytick.color'] = 'gray'
plt.rcParams['xtick.color'] = 'gray'
plt.rcParams['ytick.color'] = 'gray'
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['xtick.labelsize'] = 8

## Helper Functions

### calculate_bounds

In [381]:
def calculate_bounds(df_1: pd.DataFrame, df_2: pd.DataFrame, recipe_choices = [], alpha = 1.0):
    """
    Calculates the estimated scores and LinUCB upper bounds for recipes.

    This function simulates a linear contextual bandit model (LinUCB)
    to estimate the scores of all recipes based on a set of chosen recipes and their
    corresponding ratings. It also computes an upper confidence bound for each recipe's
    score, which balances exploitation (choosing recipes with high estimated scores)
    and exploration (choosing recipes with high uncertainty).

    Args:
        df_1 (pd.DataFrame): A DataFrame of recipe features. The index should be
                             the recipe slugs and columns should be the features (e.g., tags).
                             This DataFrame should include a 'bias' column.
        df_2 (pd.DataFrame): A DataFrame of recipe ratings. The index should be the
                             recipe slugs and it must contain a 'rating' column.
        recipe_choices (list): A list of recipe slugs that have been "chosen" or
                               rated so far. The model will be trained on the data
                               corresponding to these choices.
        alpha (float): The exploration parameter for the LinUCB model. A higher
                       value of alpha increases the model's focus on exploring
                       recipes with higher uncertainty (wider confidence bounds).

    Returns:
        pd.DataFrame: A DataFrame with the following columns:
                      - 'score_estimate': The estimated score for each recipe based on
                                         the linear regression model.
                      - 'score_bound': The LinUCB upper confidence bound for each
                                       recipe's score.
                      - 'num_features': The number of features (tags) for each recipe.
                      The index of the DataFrame is the recipe slugs.
    """
    recipe_choices = list(recipe_choices)

    D = df_1.loc[recipe_choices].to_numpy(dtype="float64")
    c = df_2.loc[recipe_choices, 'rating'].to_numpy(dtype="float64")

    DTDI = np.eye(D.shape[1])
    if len(recipe_choices) > 0:
        DTDI += D.T @ D
    DTDI_inv = np.linalg.inv(DTDI)

    theta_hat       = DTDI_inv @ D.T @ c
    features_array  = df_1.to_numpy()
    means           = features_array @ theta_hat

    variances = []
    for z in features_array:
        z = z.reshape(-1, 1)
        variances.append((z.T @ DTDI_inv @ z).item())
    variances = np.array(variances)

    df = pd.DataFrame({"score_estimate": means,
    "score_bound": means + alpha * np.sqrt(variances)}, index=df_1.index)
    df["num_features"] = df_1.T.sum().T

    return df

# calculate_bounds().sort_values("score_bound", ascending=False)

### try_picks

In [382]:
def try_picks(features_df, ratings_df, alpha = 2.0, num_picks = 100):
    """
    Simulates an online learning process with LinUCB to make a number of recommendations.

    Args:
        features_df (pd.DataFrame): A DataFrame of recipe features with 'recipe_slug' as index.
        ratings_df (pd.DataFrame): A DataFrame of recipe ratings with 'recipe_slug' as index.
        alpha (float): The exploration parameter for the LinUCB model.
        num_picks (int): The number of recommendations to make.

    Returns:
        pd.DataFrame: A DataFrame containing the recommendations, their score bounds, and rewards.
    """
    recipe_choices  = []
    recommendations = []
    
    
    for i in range(num_picks):
        # Calculate bounds based on the current set of chosen recipes
        current_bounds = calculate_bounds(features_df, ratings_df, recipe_choices = recipe_choices, alpha = alpha)

        # recipe with highest upper confidence bound
        best_bound      = current_bounds["score_bound"].max()
        best_recipes    = current_bounds[current_bounds["score_bound"] == best_bound].copy()
        
        # tiebreaker through random choice
        choice              = best_recipes.sample(1)
        chosen_recipe_slug  = choice.index[0]

        # true rating (reward) for chosen recipe
        reward = ratings_df.loc[chosen_recipe_slug, 'rating']

        # store recommendation details
        recommendations.append({
            'recipe_slug': chosen_recipe_slug,
            'score_bound': choice.loc[chosen_recipe_slug, 'score_bound'],
            'reward': reward
        })

        # Add chosen recipe slug to list for next iteration
        recipe_choices.append(chosen_recipe_slug)
        
    return pd.DataFrame(recommendations)

## Part 1: Collect Rating Data

The file **recipes.tsv** in this repository has information about 100 recipes.
- Make a new file "**ratings.tsv**" with two columns, `recipe_slug` (from recipes.tsv) and `rating`.
- Populate the `rating` column with values between 0 and 1 where 0 is the worst and 1 is the best. You can assign these ratings however you want within that range, but try to make it reflect a consistent set of preferences. These could be your preferences, or a persona of your choosing (e.g. chocolate lover, bacon-obsessed, or sweet tooth).
- Make sure that there are at least 10 ratings of zero and at least 10 ratings of one.


Hint: You may find it more convenient to assign raw ratings from 1 to 5 and then remap them as follows.

`ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25`

Submit "ratings.tsv" in Gradescope.

In [383]:
# YOUR CHANGES HEREw

# Load  datasets 

recipes     = 'https://raw.githubusercontent.com/bu-cds-dx704/dx704-project-04/main/recipes.tsv'
recipe_tags = 'https://raw.githubusercontent.com/bu-cds-dx704/dx704-project-04/main/recipe-tags.tsv'

try:
    recipes_df      = pd.read_csv(recipes, sep = '\t')
    recipe_tags_df  = pd.read_csv(recipe_tags, sep = '\t')
except FileNotFoundError:
    print("Error: Make sure 'recipes.tsv' and 'recipe-tags.tsv' are in the same directory as this notebook.")
    # Exit or raise an exception to stop execution
   
    recipes_df      = pd.DataFrame(columns=['recipe_slug']) # Create empty df to avoid immediate crash



In [384]:
recipes_df

,recipe_slug,recipe_title,recipe_introduction
0,falafel,Falafel,Falafel is a popular Middle Eastern dish made ...
1,spamburger,Spamburger,Spamburger is a type of hamburger that is made...
2,bacon-fried-rice,Bacon Fried Rice,Bacon fried rice is a savory and satisfying di...
3,chicken-fingers,Chicken Fingers,Chicken fingers are a popular dish made from c...
4,apple-crisp,Apple Crisp,Apple crisp is a classic dessert made with bak...
...,...,...,...
95,bacon-mac-and-cheese,Bacon Mac And Cheese,Bacon mac and cheese is a delicious and comfor...
96,chicken-alfredo-lasagna,Chicken Alfredo Lasagna,Chicken alfredo lasagna is a delicious twist o...
97,classic-beef-lasagna,Classic Beef Lasagna,Classic beef lasagna is a hearty and comfortin...
98,vegetarian-mushroom-lasagna,Vegetarian Mushroom Lasagna,Vegetarian mushroom lasagna is a delicious and...


In [385]:
recipe_tags_df

,recipe_slug,recipe_tag
0,spam-musubi,hawaiian
1,spam-musubi,nori
2,spam-musubi,onthego
3,spam-musubi,rice
4,spam-musubi,snack
...,...,...
747,bacon-souffle,breakfast
748,bacon-souffle,brunch
749,bacon-souffle,cheese
750,bacon-souffle,eggs


In [386]:
# YOUR CHANGES HERE

# Sweet tooth persona ! 
sweet_tags = {
    'dessert', 'chocolate', 'cinnamon', 'chocolatechip', 'cake', 'cookies', 'brownies', 'sweet', 'icecream',
    'cobbler', 'crisp', 'crumble', 'pie', 'cupcakes', 'babka', 'pastry', 'sugar', 'brownsugar', 'powdersugar',
    'frosting', 'ganache', 'vanilla', 'almond', 'peanutbutter', 'fruit', 'caramel', 'pudding', 'mousse', 'tart', 'fudge', 'baking', 'custard'
}


In [387]:
testing = recipe_tags_df.groupby('recipe_slug')['recipe_tag'].apply(list)
testing

recipe_slug
almond-chip-cookies            [almond, baking, chocolatechip, cookies, dessert]
almond-croissants              [almond, breakfast, brunch, buttery, flaky, fr...
apple-crisp                    [apple, baked, comfortfood, crisp, crumble, de...
apple-crumble                  [apple, baked, brownsugar, buttery, cinnamon, ...
apple-pie                      [apple, baking, cinnamon, cloves, crust, desse...
                                                     ...                        
tempura-udon                   [comfortfood, dashibroth, japanesecuisine, tem...
udon                           [glutenfree, japanesecuisine, noodles, soup, v...
vanilla-souffle                [custard, dessert, eggwhites, french, souffle,...
vegetable-lasagna              [comfortfood, healthy, italian, pasta, vegetar...
vegetarian-mushroom-lasagna    [comfortfood, italian, lasagna, mushroom, past...
Name: recipe_tag, Length: 100, dtype: object

In [388]:

# Count sweet tags for each recipe
tag_counts = recipe_tags_df.groupby('recipe_slug')['recipe_tag'].apply(list).reset_index()
tag_counts

,recipe_slug,recipe_tag
0,almond-chip-cookies,"[almond, baking, chocolatechip, cookies, dessert]"
1,almond-croissants,"[almond, breakfast, brunch, buttery, flaky, fr..."
2,apple-crisp,"[apple, baked, comfortfood, crisp, crumble, de..."
3,apple-crumble,"[apple, baked, brownsugar, buttery, cinnamon, ..."
4,apple-pie,"[apple, baking, cinnamon, cloves, crust, desse..."
...,...,...
95,tempura-udon,"[comfortfood, dashibroth, japanesecuisine, tem..."
96,udon,"[glutenfree, japanesecuisine, noodles, soup, v..."
97,vanilla-souffle,"[custard, dessert, eggwhites, french, souffle,..."
98,vegetable-lasagna,"[comfortfood, healthy, italian, pasta, vegetar..."


In [389]:
sweet_scores = []
for tag_list in tag_counts['recipe_tag']:
    score = 0
    for tag in tag_list:
        if tag in sweet_tags:
            score += 1
    sweet_scores.append(score)

tag_counts['sweet_score'] = sweet_scores
print("tag_counts with sweet scores:")
tag_counts

tag_counts with sweet scores:


,recipe_slug,recipe_tag,sweet_score
0,almond-chip-cookies,"[almond, baking, chocolatechip, cookies, dessert]",5
1,almond-croissants,"[almond, breakfast, brunch, buttery, flaky, fr...",3
2,apple-crisp,"[apple, baked, comfortfood, crisp, crumble, de...",3
3,apple-crumble,"[apple, baked, brownsugar, buttery, cinnamon, ...",5
4,apple-pie,"[apple, baking, cinnamon, cloves, crust, desse...",5
...,...,...,...
95,tempura-udon,"[comfortfood, dashibroth, japanesecuisine, tem...",0
96,udon,"[glutenfree, japanesecuisine, noodles, soup, v...",0
97,vanilla-souffle,"[custard, dessert, eggwhites, french, souffle,...",3
98,vegetable-lasagna,"[comfortfood, healthy, italian, pasta, vegetar...",0


In [390]:
# tag_counts['sweet_score'] = tag_counts['recipe_tag'].apply(lambda tags: sum(tag in sweet_tags for tag in tags))


In [391]:

# Merge sweet scores with recipes ... using merge to ensure all recipes are included although it looks like every recipe received a tag from the sweet tooth persona
ratings_df = recipes_df[['recipe_slug']].merge(tag_counts[['recipe_slug', 'sweet_score']], on = 'recipe_slug', how = 'left')
ratings_df


,recipe_slug,sweet_score
0,falafel,0
1,spamburger,0
2,bacon-fried-rice,0
3,chicken-fingers,0
4,apple-crisp,3
...,...,...
95,bacon-mac-and-cheese,0
96,chicken-alfredo-lasagna,0
97,classic-beef-lasagna,0
98,vegetarian-mushroom-lasagna,0


In [392]:
ratings_df['sweet_score'] = ratings_df['sweet_score'].fillna(0)

# Assign raw ratings based on sweet_score
def score_to_rating(score):
    if score == 0:
        return 1
    elif score == 1:
        return 2
    elif score == 2:
        return 3
    elif score == 3:
        return 4
    else:  # score >= 3
        return 5

ratings_df['rating_raw'] = ratings_df['sweet_score'].apply(score_to_rating)
ratings_df


,recipe_slug,sweet_score,rating_raw
0,falafel,0,1
1,spamburger,0,1
2,bacon-fried-rice,0,1
3,chicken-fingers,0,1
4,apple-crisp,3,4
...,...,...,...
95,bacon-mac-and-cheese,0,1
96,chicken-alfredo-lasagna,0,1
97,classic-beef-lasagna,0,1
98,vegetarian-mushroom-lasagna,0,1


In [393]:
# Remap raw ratings to 0-1 scale
ratings_df['rating'] = (ratings_df['rating_raw'] - 1) * 0.25
ratings_df

,recipe_slug,sweet_score,rating_raw,rating
0,falafel,0,1,0.00
1,spamburger,0,1,0.00
2,bacon-fried-rice,0,1,0.00
3,chicken-fingers,0,1,0.00
4,apple-crisp,3,4,0.75
...,...,...,...,...
95,bacon-mac-and-cheese,0,1,0.00
96,chicken-alfredo-lasagna,0,1,0.00
97,classic-beef-lasagna,0,1,0.00
98,vegetarian-mushroom-lasagna,0,1,0.00


In [394]:
# filter dataframe to only include recipes with rating = 1
# test_df = ratings_df.copy()
# test_df = test_df[test_df['rating_raw'] == 5]
# test_df


In [395]:
ratings_df_final = ratings_df[['recipe_slug', 'rating']]

# sort by recipe_slug to ensure consistent ordering
ratings_df_final = ratings_df_final.sort_values(by = 'recipe_slug').reset_index(drop = True)
ratings_df_final

,recipe_slug,rating
0,almond-chip-cookies,1.00
1,almond-croissants,0.75
2,apple-crisp,0.75
3,apple-crumble,1.00
4,apple-pie,1.00
...,...,...
95,tempura-udon,0.00
96,udon,0.00
97,vanilla-souffle,0.75
98,vegetable-lasagna,0.00


In [396]:
ratings_df_final.to_csv("ratings.tsv", sep="\t", index = False)

print("ratings.tsv file created successfully.")

ratings.tsv file created successfully.


## Part 2: Construct Model Input

Use your file "ratings.tsv" combined with "recipe-tags.tsv" to create a new file "features.tsv" with a column `recipe_slug`, a column `bias` which is hard-coded to one, and a column for each tag that appears in "recipe-tags.tsv".
The tag column in this file should be a 0-1 encoding of the recipe tags for each recipe.
[Pandas reshaping function methods](https://pandas.pydata.org/docs/user_guide/reshaping.html) may be helpful.

The bias column will make later LinUCB calculations easier since it will just be another dimension.

Hint: For later modeling steps, it will be important to have the feature data (inputs) and the rating data (target outputs) in the same order.
It is highly recommended to make sure that "features.tsv" and "ratings.tsv" have the recipe slugs in the same order.

In [397]:
# YOUR CHANGES HERE

recipe_tags_df

,recipe_slug,recipe_tag
0,spam-musubi,hawaiian
1,spam-musubi,nori
2,spam-musubi,onthego
3,spam-musubi,rice
4,spam-musubi,snack
...,...,...
747,bacon-souffle,breakfast
748,bacon-souffle,brunch
749,bacon-souffle,cheese
750,bacon-souffle,eggs


In [398]:

tag_dummies = pd.get_dummies(recipe_tags_df['recipe_tag'])
# tag_dummies


In [399]:
features_df = recipe_tags_df.copy()
features_df = features_df[['recipe_slug']].join(tag_dummies)
# features_df

In [400]:

 # Group by recipe_slug to get one row per recipe
features_df = features_df.groupby('recipe_slug').sum().reset_index()
# features_df

In [401]:
tag_cols = features_df.columns.tolist()
tag_cols.remove('recipe_slug')
# tag_cols

In [402]:
features_df['bias'] = 1
cols                = ['recipe_slug','bias'] + tag_cols
features_df         = features_df[cols]
features_df


,recipe_slug,bias,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus,...,udonnoodles,vanilla,vanillaicecream,vegan,vegetables,vegetarian,warm,whippedcream,winter,yeastdough
0,almond-chip-cookies,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,almond-croissants,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,apple-crisp,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
3,apple-crumble,1,0,0,0,0,0,1,0,0,...,0,0,1,0,0,0,0,1,0,0
4,apple-pie,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,tempura-udon,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
96,udon,1,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
97,vanilla-souffle,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
98,vegetable-lasagna,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [403]:

# Save to features.tsv
features_df.to_csv("features.tsv", sep="\t", index=False)

print("features.tsv file has been created successfully.")



features.tsv file has been created successfully.


Submit "features.tsv" in Gradescope.

## Part 3: Linear Preference Model

Use your feature and rating files to build a **ridge regression model** with ridge regression's regularization parameter $\alpha$ set to 1.


Hint: If you are using scikit-learn modeling classes, you should use `fit_intercept = False` since that intercept value will be redundant with the bias coefficient.

Hint: The estimate component of the bounds should match the previous estimate, so you should be able to just focus on the variance component of the bounds now.

In [404]:
# YOUR CHANGES HERE

model_df = pd.merge(features_df, ratings_df_final, on='recipe_slug')
model_df

,recipe_slug,bias,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus,...,vanilla,vanillaicecream,vegan,vegetables,vegetarian,warm,whippedcream,winter,yeastdough,rating
0,almond-chip-cookies,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1.00
1,almond-croissants,1,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.75
2,apple-crisp,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0.75
3,apple-crumble,1,0,0,0,0,0,1,0,0,...,0,1,0,0,0,0,1,0,0,1.00
4,apple-pie,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,tempura-udon,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.00
96,udon,1,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0.00
97,vanilla-souffle,1,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0.75
98,vegetable-lasagna,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0.00


In [405]:

X       = model_df.drop(columns = ["recipe_slug", "rating"])
y       = model_df["rating"]
alpha   = 1.0  # Regularization strength


model   = Ridge(alpha = alpha, fit_intercept = False) 
model.fit(X, y)

y_pred  = model.predict(X)
mse     = mean_squared_error(y, y_pred)
rmse    = np.sqrt(mse)
print(f"RMSE: {rmse:.4f}")


coefficients    = model.coef_
feature_names   = X.columns

coef_df = pd.DataFrame({
    "recipe_tag": feature_names,
    "coefficient": coefficients
})

coef_df


RMSE: 0.0229


,recipe_tag,coefficient
0,bias,0.110235
1,alfredo,-0.011821
2,almond,0.130509
3,american,0.010168
4,appetizer,-0.035425
...,...,...
292,vegetarian,-0.041656
293,warm,-0.001047
294,whippedcream,0.005121
295,winter,0.031472


Save the coefficients of this model in a file "model.tsv" with columns "recipe_tag" and "coefficient".
Do not include the bias.

In [406]:
# YOUR CHANGES HERE

# coef_df = coef_df[coef_df["recipe_tag"] != "bias"]

coef_df.to_csv("model.tsv", sep="\t", index=False)


Submit "model.tsv" in Gradescope.

## Part 4: Recipe Estimates

Use the recipe model to estimate the score of every recipe.
Save these estimates to a file "estimates.tsv" with columns `recipe_slug` and `score_estimate`.

In [407]:
# YOUR CHANGES HERE


score_estimates = model.predict(X)

estimates_df = pd.DataFrame({
    "recipe_slug": features_df["recipe_slug"],
    "score_estimate": score_estimates
})
estimates_df

,recipe_slug,score_estimate
0,almond-chip-cookies,0.959941
1,almond-croissants,0.669368
2,apple-crisp,0.752303
3,apple-crumble,0.994879
4,apple-pie,1.002480
...,...,...
95,tempura-udon,0.017730
96,udon,-0.000072
97,vanilla-souffle,0.721026
98,vegetable-lasagna,-0.004332


Submit "estimates.tsv" in Gradescope.

In [408]:
estimates_df.to_csv("estimates.tsv", sep="\t", index=False)

## Part 5: LinUCB Bounds

Calculate the upper bounds of LinUCB using data corresponding to trying every recipe once and receiving the rating in "ratings.tsv" as the reward.
Keep the ridge regression regularization parameter at 1, and set LinUCB's $\alpha$ parameter to 2.
Save these upper bounds to a file "bounds.tsv" with columns `recipe_slug` and `score_bound`.

In [409]:
#  calculate_bounds function requires recipe_choices to be the list of recipes that have been selected.

#  entire list of recipe slugs 
all_recipe_slugs = features_df['recipe_slug'].tolist()
# all_recipe_slugs

In [410]:
ratings_df_reindexed = ratings_df_final.copy().set_index('recipe_slug')
# ratings_df_reindexed

In [411]:
features_df_reindexed = features_df.copy().set_index('recipe_slug')
# features_df_reindexed

In [412]:
alpha_linucb    = 2.0
bounds_df       = calculate_bounds(features_df_reindexed, 
                                   ratings_df_reindexed, 
                                   recipe_choices   = all_recipe_slugs, 
                                   alpha            = alpha_linucb
                                   )

bounds_df


,score_estimate,score_bound,num_features
recipe_slug,,,
almond-chip-cookies,0.959941,2.683399,6
almond-croissants,0.669368,2.375223,9
apple-crisp,0.752303,2.528645,11
apple-crumble,0.994879,2.864551,14
apple-pie,1.002480,2.852732,11
...,...,...,...
tempura-udon,0.017730,1.811604,6
udon,-0.000072,1.769669,7
vanilla-souffle,0.721026,2.524222,7


In [413]:
bounds_df_final = bounds_df.copy().reset_index()
bounds_df_final = bounds_df_final[['recipe_slug', 'score_bound']]
bounds_df_final

,recipe_slug,score_bound
0,almond-chip-cookies,2.683399
1,almond-croissants,2.375223
2,apple-crisp,2.528645
3,apple-crumble,2.864551
4,apple-pie,2.852732
...,...,...
95,tempura-udon,1.811604
96,udon,1.769669
97,vanilla-souffle,2.524222
98,vegetable-lasagna,1.462540


In [414]:
bounds_df_final.to_csv("bounds.tsv", sep = "\t", index = False)

print("LinUCB upper bounds saved to bounds.tsv.")


LinUCB upper bounds saved to bounds.tsv.


Submit "bounds.tsv" in Gradescope.

## Part 6: Make Online Recommendations

Implement LinUCB to make 100 recommendations starting with no data and using the same parameters as in part 5.
- One recommendation should be made at a time and you can break ties arbitrarily.
- After each recommendation, use the rating from part 1 as the reward to update the LinUCB data.
- Record the recommendations made in a file "recommendations.tsv" with columns `recipe_slug`, `score_bound`, and `reward`.
- The rows in this file should be in the same order as the recommendations were made.

In [415]:
# YOUR CHANGES HERE

recommendations_df = try_picks(features_df_reindexed, ratings_df_reindexed, alpha = 2.0, num_picks = 100)
recommendations_df


,recipe_slug,score_bound,reward
0,apple-crumble,7.483315,1.00
1,ramen,7.270093,0.00
2,quesadillas,7.235617,0.00
3,ma-la-chicken,7.095599,0.00
4,chocolate-babka,6.936422,0.75
...,...,...,...
95,butter-croissants,2.874694,0.50
96,apple-crumble,2.864574,1.00
97,strawberry-rhubarb-crisp,2.855215,1.00
98,apple-pie,2.852069,1.00


In [416]:
recommendations_df.to_csv("recommendations.tsv", sep = "\t", index = False)
print("Recommendations saved to recommendations.tsv.")

Recommendations saved to recommendations.tsv.


Submit "recommendations.tsv" in Gradescope.

## Part 7: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgements are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.


Submit "project.ipynb" in Gradescope.

## Notes

<font color = 'plum'>
Part 5 and Part 6 represent two fundamentally different approaches to using the LinUCB model.


Part 5 is a "batch" calculation. It simulates a scenario where you have already tried every single recipe once. The recipe_choices list is populated with all 100 recipes, and calculate_bounds is run a single time to produce a final set of bounds. This is a retrospective analysis to see what the bounds would be after all data has been collected.

Part 6 is an "online" or "sequential" learning process, which is the primary use case for a bandit algorithm. The goal is to start with no information (an empty recipe_choices list) and make decisions one step at a time. The try_picks function is designed for this exact purpose. In each iteration of its loop, it uses the data from the recipes chosen so far to calculate new bounds. It then makes one new recommendation based on those new bounds, and that new recommendation is added to the training data for the next iteration.

If Part 6 were to simply load the bounds.tsv from Part 5, it would skip the entire learning process and assume it already has all the information. The purpose of this part of the project is to demonstrate how the bandit algorithm learns over time, which requires the iterative loop that the try_picks function provides.